# Partie 2 : session pilote pass@k (v2)

50 questions de ponts HotpotQA (tirage de mise au point, graine 7, disjoint des questions d'évaluation), 32 tirages par
question, au plus 4 recherches, température 0,7. Objectif : mesurer le **débit**, le **taux de bon format** et le
**taux de troncature** de chaque modèle, pour fixer la taille de l'étude (`docs/partie2/plan_analyse.md`).

Deux files de runs en parallèle, une par GPU T4. Un run qui échoue n'arrête pas les autres. Sorties :
`results/partie2/`, `logs/`, `resume_pilote.json`.

In [ ]:
BRANCH = "partie2-passk"
N_QUESTIONS, N_SAMPLES, T, TEMP, SEED_QUESTIONS = 50, 32, 4, 0.7, 7
# Pilote v2 : 1 024 jetons par tour (28,5 % de troncature à 512 pour l'agent Instruct GRPO en v1),
# contexte de 8 192 jetons (4 096 faisait planter les lots en v1).
MAX_TOKENS, MAX_MODEL_LEN = 1024, 8192
QWEN_IT, QWEN_BASE = "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-3B"
R = "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-"
# (modèle, options) ; les deux files tournent en parallèle, les runs essentiels d'abord.
FILES = [
    [(R + "it-em-grpo-v0.3", []), (R + "it-em-ppo-v0.3", []), (QWEN_BASE, ["--no-chat-template"])],
    [(QWEN_IT, []), (QWEN_BASE, []), (R + "em-ppo-v0.3", [])],
]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
from pathlib import Path
REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/tmp/memoire-rl-search")
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm==0.33.0 pandas pyarrow huggingface_hub 2>&1 | tail -3
!pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"
import sys
sys.path.insert(0, str(CLONE))
from data.hotpot_distractor import telecharger
PARQUET = telecharger("/tmp/hotpot")
print(PARQUET)

In [ ]:
import os, threading, time
WORK = Path("/kaggle/working")
(WORK / "logs").mkdir(exist_ok=True)
status, verrou = [], threading.Lock()

def file_de_runs(gpu, runs):
    env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn", CUDA_VISIBLE_DEVICES=str(gpu))
    for model, opts in runs:
        nom = model.split("/")[-1] + ("_brut" if "--no-chat-template" in opts else "")
        cmd = ["python", "-u", "scripts/run_passk.py", "--model", model, "--type", "bridge",
               "--n-questions", str(N_QUESTIONS), "--n-samples", str(N_SAMPLES), "--max-turns", str(T),
               "--temperature", str(TEMP), "--seed-questions", str(SEED_QUESTIONS), "--data", PARQUET,
               "--max-tokens", str(MAX_TOKENS), "--max-model-len", str(MAX_MODEL_LEN),
               "--outdir", str(WORK / "results" / "partie2")] + opts
        t0 = time.time()
        with open(WORK / "logs" / f"{nom}.log", "a") as f:
            code = subprocess.run(cmd, cwd=CLONE, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        with verrou:
            status.append((nom, gpu, code, round(time.time() - t0)))
            print(f"GPU {gpu} : {nom} code {code}, {time.time() - t0:.0f} s", flush=True)

fils = [threading.Thread(target=file_de_runs, args=(g, runs)) for g, runs in enumerate(FILES)]
for f in fils: f.start()
for f in fils: f.join()

In [ ]:
import json
from collections import Counter
resume = {}
for fichier in sorted((WORK / "results" / "partie2").glob("*.jsonl")):
    L = [json.loads(l) for l in fichier.read_text().split("\n") if l.strip()]
    if not L:
        continue
    st = Counter(l["status"] for l in L)
    par_q = Counter(l["id"] for l in L)
    resume[fichier.stem] = {
        "trajectoires": len(L), "questions_completes": sum(v >= N_SAMPLES for v in par_q.values()),
        "em": round(sum(l["em"] for l in L) / len(L), 3),
        "format_ok": round(1 - (st["format"] + st["truncated"] + st["context_exceeded"]) / len(L), 3),
        "statuts": dict(st), "tronquees": round(sum(l["truncated"] for l in L) / len(L), 3),
        "recherches": round(sum(l["n_search"] for l in L) / len(L), 2),
        "jetons": round(sum(l["gen_tokens"] for l in L) / len(L)),
    }
for nom, gpu, code, secondes in status:
    # même nom court que scripts/run_passk.py, et correspondance exacte (Qwen2.5-3B n'est pas Qwen2.5-3B-Instruct)
    court = nom.replace("_brut", "").replace("SearchR1-nq_hotpotqa_train-", "")
    tag = "brut" if nom.endswith("_brut") else "chat"
    attendu = f"{court}_bridge_q{N_QUESTIONS}_n{N_SAMPLES}_T{T}_temp{TEMP}_{tag}_seed{SEED_QUESTIONS}"
    if attendu in resume:
        resume[attendu]["secondes"] = secondes
        resume[attendu]["traj_par_s"] = round(resume[attendu]["trajectoires"] / max(secondes, 1), 2)
(WORK / "resume_pilote.json").write_text(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
print(json.dumps(resume, indent=1, ensure_ascii=False))
echecs = [s for s in status if s[2] != 0]
if echecs:
    raise RuntimeError(f"Runs en échec : {echecs}")